<a href="https://colab.research.google.com/github/PpaulaPulido/DiplomadoIA/blob/main/regresion_clasificacion_para_completar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Versión para completar.** Repaso de **regresión y clasificación**.
> Solución: `07_repaso_regresion_clasificacion.ipynb`.
> CSV: `Penguins.csv` en esta carpeta.

# UDECATALUNA — para completar
_________________________________

Tema: Modelos de Aprendizaje — **Repaso**: regresión y clasificación
_________________________________

**Dataset:** `Penguins.csv` (el mismo para los dos problemas)  
**Regresión:** `LinearRegression` → predecir el **peso**  
**Clasificación:** `DecisionTreeClassifier` → predecir la **especie**

## Objetivo

Al terminar vas a poder:

* Decir en una frase qué pregunta es de **regresión** y cuál de **clasificación**.
* Entrenar un modelo de cada tipo, con baseline, `Pipeline` y test al final.
* Leer **pocas** métricas: MAE y R² en regresión; accuracy, matriz y reporte en clasificación.
* Resolver 12 ejercicios (las celdas de código están vacías).

> La tabla no decide el problema. **El target sí.**  
> Mismas filas, dos preguntas distintas.

**Reglas (las de siempre)**

1. No entrenamos con el dataset entero.
2. El preproceso vive dentro del `Pipeline`.
3. Primero un **baseline**, después el modelo.
4. El **test** se mira al final.
5. El código va **seguido**. Sin `armar_modelo()`.

| | Regresión | Clasificación |
|---|---|---|
| Pregunta | ¿cuánto pesa? | ¿qué especie es? |
| Target | `peso` (gramos, número) | `especie` (3 etiquetas) |
| Modelo | recta: $\hat{y} = w\cdot x + b$ | árbol: preguntas sí/no |
| Baseline | predecir la **media** del train | predecir la clase **más frecuente** |
| Métricas | MAE, R² | accuracy, matriz, reporte |

In [ ]:
# PASO 1 — Entorno
# warnings, plt, np, pd, sns, display. whitegrid. versión de pandas.
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context = "notebook")
plt.rcParams["figure.figsize"] = (10, 4)

print("pandas:", pd.__version__)

pandas: 2.2.3


### El mismo CSV, dos problemas

| Columna | Qué es | En regresión | En clasificación |
|---|---|---|---|
| `pico_largo`, `pico_ancho`, `aleta` | medidas | features | features |
| `isla`, `sexo` | categorías | features | features |
| `peso` | gramos | **target** | feature |
| `especie` | Adelie / Chinstrap / Gentoo | no entra (fuga: el peso ya “sabe” la especie un poco) | **target** |

Si en la regresión metés `especie`, estás soplando: Gentoo pesa más. El biólogo que pesa al animal **todavía no** necesita la especie; al revés, a veces pesa para ayudar a clasificar.

### Bloque 1 — Cargar y limpiar (una vez)

In [ ]:
data = pd.read_csv("/content/Penguins.csv")
data.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [ ]:
df = data.dropna().copy()
df = df.rename(columns={
    "species": "especie",
    "island": "isla",
    "bill_length_mm": "pico_largo",
    "bill_depth_mm": "pico_ancho",
    "flipper_length_mm": "aleta",
    "body_mass_g": "peso",
    "sex": "sexo",
})
display(df.head())

,especie,isla,pico_largo,pico_ancho,aleta,peso,sexo
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE


In [ ]:
display(df["especie"].value_counts())

,count
especie,
Adelie,146
Gentoo,119
Chinstrap,68


### Bloque 2 — Una mirada (todavía sin modelo)

Izquierda: el peso es un **número** (regresión).  
Derecha: la especie es una **etiqueta** (clasificación).

> **Para discutir.**  
> 1. Si solo te dan la aleta, ¿podés **adivinar el peso** (un número) o la **especie** (una de tres)? Las dos, con error distinto.  
> 2. ¿Por qué no numerar Adelie=1, Chinstrap=2, Gentoo=3 y hacer una recta?  
> 3. En la clasificación, ¿dejarías `peso` como feature? (sí: el biólogo lo midió). En la regresión, ¿dejarías `especie`? (no: es casi la respuesta).

# Parte A — Regresión: predecir el peso

**Target:** `peso`  
**Features:** pico, aleta, isla, sexo. **No** `especie`.

Métricas de esta parte (solo dos):

* **MAE** — error absoluto medio, en gramos. “Me equivoco, en promedio, tantos gramos.”
* **R²** — 1 es perfecto, 0 es “igual que predecir la media”. Negativo = peor que el loro.

In [ ]:
df["isla"].value_counts()

,count
isla,
Biscoe,163
Dream,123
Torgersen,47


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor # libreria para generar un comparativo sin modelo
from sklearn.linear_model import LinearRegression # libreria del modelo
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_r = ["pico_largo",	"pico_ancho",	"aleta"]
cat_r = ["isla", "sexo"]

xr = df[num_r + cat_r]
yr = df["peso"]

xr_train, xr_test, yr_train, yr_test = train_test_split(
    xr, yr, test_size=0.2, random_state=42
)
print("Train", len(xr_train), "| Test:", len(xr_test))

Train 266 | Test: 67


Baseline: `DummyRegressor(strategy="mean")` — siempre dice la media del train.  
Modelo: one-hot + scaler + `LinearRegression`. El scaler acá sí importa (gramos vs mm).

In [ ]:
dummy_r = DummyRegressor(strategy="mean")
dummy_r.fit(xr_train, yr_train)

prep_r = ColumnTransformer([
    ("num", StandardScaler(), num_r),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_r)
])
reg = Pipeline([
    ("prep", prep_r),
    ("modelo", LinearRegression())
])
reg.fit(xr_train, yr_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['pico_largo', 'pico_ancho',
                                                   'aleta']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['isla', 'sexo'])])),
                ('modelo', LinearRegression())])

In [ ]:
pred_dummy_r = dummy_r.predict(xr_test)
pred_r = reg.predict(xr_test)
print("MAE dummy (estrategia asignarle el promedio del peso al valor nuevo):", round(mean_absolute_error(yr_test, pred_dummy_r), 2))
print("MAE modelo regresion:", round(mean_absolute_error(yr_test, pred_r), 2))
print("R² dummy:", round(r2_score(yr_test, pred_dummy_r), 2))
print("R² modelo regresion:", round(r2_score(yr_test, pred_r), 2))

MAE dummy (estrategia asignarle el promedio del peso al valor nuevo): 674.13
MAE modelo regresion: 235.86
R² dummy: -0.0
R² modelo regresion: 0.87


# Parte B — Clasificación: predecir la especie

**Target:** `especie`  
**Features:** pico, aleta, peso, isla, sexo.

Métricas de esta parte (las tres de siempre): accuracy vs dummy, matriz, reporte.

Baseline: siempre Adelie.  
Árbol `max_depth=3`. **No** scaler: el árbol corta en las unidades crudas.

# Parte C — Un pingüino, dos preguntas

Misma fila: ¿cuánto pesa? ¿qué especie es?

### Ejercicios (para completar)

1–5 y 11 regresión. 6–10 y 12 clasificación. Solución: `07_repaso_regresion_clasificacion.ipynb`.

#### Ejercicio 1 — ¿Cuánto mejora la recta?

MAE_loro − MAE_modelo, en gramos. Esa es la mejora media.

#### Ejercicio 2 — Un pingüino liviano y uno pesado

Dos filas. ¿La recta respeta que más aleta / más pico → más peso?

#### Ejercicio 3 — Regresión solo con la aleta

Una columna. ¿El MAE se parece al modelo full o se va hacia el loro?

In [ ]:
# EJ 3 — LinearRegression solo con aleta: MAE vs full vs loro


#### Ejercicio 4 — La trampa: usar `especie` para predecir el peso

Reentrená **incluyendo** especie. El MAE baja. Eso no es un mérito: es soplar.

#### Ejercicio 5 — Por qué no es clasificación

El peso 3700 no es una clase. Si recortás “liviano / pesado” con la mediana, **ahí sí** sería clasificación (como en Sydney). Mientras el target sea el gramo, es regresión.

#### Ejercicio 6 — Recall de Chinstrap a mano

In [ ]:
# EJ 6 — crosstab y recall de Chinstrap a mano


#### Ejercicio 7 — Dummy estratificado vs siempre Adelie vs 1/3

In [ ]:
# EJ 7 — Dummy stratified vs most_frequent vs 1/3 vs árbol


#### Ejercicio 8 — `max_depth` 1, 3 y 10 (train vs test)

In [ ]:
# EJ 8 — max_depth 1, 3, 10: accuracy train y test


#### Ejercicio 9 — La trampa inversa: clasificar con una recta sobre 1, 2, 3

Numerar especies y usar `LinearRegression` es **el error de esta materia**.  
La recta asume que Chinstrap está 'en el medio'. Mostralo y descartalo.

In [ ]:
# EJ 9 — Recta sobre Adelie=1, Chinstrap=2, Gentoo=3. Compará accuracy. Mostrá yhat crudo.


#### Ejercicio 10 — Ficha de un pingüino del test

Tomá la **primera fila del test de clasificación**.  
Decí: especie real, especie predicha, y el peso que diría el modelo de regresión (sin usar especie).

In [ ]:
# EJ 10 — Primera fila del test: real, árbol, peso de la recta, peso real


#### Ejercicio 11 — Residuos: ¿cuántos se van más de 400 g?

El MAE es un promedio. Contá cuántos pingüinos del test tienen `|peso real − predicho| > 400`. Eso muestra la cola del error, no solo el centro.


In [ ]:
# EJ 11 — residuos de la recta: cuántos |error| > 400 g, histograma, MAE y error máximo


#### Ejercicio 12 — Clasificar sin el peso

El peso ayuda a separar Gentoo (pesan más). Reentrená el mismo árbol **sin** `peso` y compará accuracy. Si casi no baja, las medidas del pico y la aleta ya alcanzaban.


In [ ]:
# EJ 12 — árbol max_depth=3 sin la columna peso; accuracy vs el modelo full


### Cierre — para decir sin notebook

* **Regresión** = ¿cuánto? Número. Baseline = media. Métricas: MAE (gramos) y R².
* **Clasificación** = ¿cuál? Etiqueta. Baseline = clase más frecuente. Métricas: accuracy, matriz, reporte.
* El dataset no define el problema. El **target** sí.
* `especie` no entra a predecir `peso`. `peso` sí puede entrar a predecir `especie`.
* No numerés las clases 1, 2, 3 para meter una recta.
* Logística y árbol **clasifican**. Lineal **regresa**. No se intercambian porque “sklearn tiene un predict”.

**UDECATALUNA** — Modelos de Aprendizaje · Repaso regresión y clasificación